In [48]:
import os
import numpy as np 
import pandas as pd
import torch
import random
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import TensorDataset, random_split
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import transforms, models
from tqdm import tqdm
from PIL import Image
import torchvision.transforms as T
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Data Visualization
import matplotlib.pyplot as plt

# Model Visualization (PyTorch không có sẵn plot_model như TF,
# có thể dùng torchsummary hoặc torchviz)
from torchsummary import summary
from torchviz import make_dot

# Extra
from typing import List, Tuple, Union

In [49]:
# Image and Mask Dimensions
IMAGE_HEIGHT = 160
IMAGE_WIDTH = 160
N_IMAGE_CHANNELS = 3
N_MASK_CHANNELS = 1

# Image and Mask Size
IMAGE_SIZE = (N_IMAGE_CHANNELS, IMAGE_HEIGHT, IMAGE_WIDTH)   # PyTorch convention: (C, H, W)
MASK_SIZE = (N_MASK_CHANNELS, IMAGE_HEIGHT, IMAGE_WIDTH)

# Batch Size and Learning Rate
BATCH_SIZE = 32
BASE_LR = 1e-2

# Model Name
MODEL_NAME = 'UNetForestSegmentation'

# Model Training
EPOCHS = 100

# Data Paths
ROOT_IMAGE_DIR = r"C:\Users\Admin\Deep Learning\Project\Forest Segmented\images"
ROOT_MASK_DIR = r"C:\Users\Admin\Deep Learning\Project\Forest Segmented\masks"
METADATA_CSV_PATH = r"C:\Users\Admin\Deep Learning\Project\Forest Segmented\meta_data.csv"

# Model Architecture
FILTERS = 32


In [50]:
# Random Seed
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [51]:
def load_image_and_mask(image_path: str, mask_path: str) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    This function takes the file paths of an image and its corresponding mask as input. 
    It first reads the images, then decodes them into tensors, 
    and resizes them to a standard size. After that, the image and mask tensors 
    are normalized by clipping the pixel values between 0 and 1. 
    Finally, the function converts the image and mask tensors to float32 
    and returns them as a tuple.

    Arguments : 
        image_path : The path to the image to be loaded. 
        mask_path  : The path to the mask to be loaded.
    
    Returns :
        image : The loaded and processed image tensor (C, H, W).
        mask  : The loaded and processed mask tensor (1, H, W).
    """

    # Read image & mask with PIL
    image = Image.open(image_path).convert("RGB")
    mask = Image.open(mask_path).convert("L")  # grayscale for mask

    # Resize
    resize = T.Resize((IMAGE_HEIGHT, IMAGE_WIDTH))
    image = resize(image)
    mask = resize(mask)

    # Convert to tensor (scales to [0,1])
    to_tensor = T.ToTensor()
    image = to_tensor(image)  # shape: (3, H, W)
    mask = to_tensor(mask)    # shape: (1, H, W)

    # Clip pixel values to [0,1]
    image = torch.clamp(image, 0.0, 1.0).float()
    mask = torch.clamp(mask, 0.0, 1.0).float()

    return image, mask

In [52]:
# Augmentation cho train set
train_transform = A.Compose([
    A.Resize(IMAGE_HEIGHT, IMAGE_WIDTH),   # resize cố định trước
    A.HorizontalFlip(p=0.5),               # lật ngang
    A.VerticalFlip(p=0.5),                 # lật dọc
    A.RandomRotate90(p=0.5),               # xoay 90 độ ngẫu nhiên
    A.Rotate(limit=30, p=0.5),             # xoay [-30, 30]
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, p=0.5), # chỉnh sáng/màu
    A.RandomResizedCrop(IMAGE_HEIGHT, IMAGE_WIDTH, scale=(0.8, 1.0), p=0.5), # crop ngẫu nhiên
    A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),  # chuẩn hóa
    ToTensorV2()   # convert sang tensor (C, H, W)
])

# Augmentation cho validation/test (chỉ resize + normalize, KHÔNG augment)
val_transform = A.Compose([
    A.Resize(IMAGE_HEIGHT, IMAGE_WIDTH),
    A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
    ToTensorV2()
])

C:\Users\Admin\AppData\Local\Temp\ipykernel_17656\3357958365.py:9: DeprecationWarning: Initializing with 'size' as an integer and a separate 'width' is deprecated. Please use a tuple (height, width) for the 'size' argument.
  A.RandomResizedCrop(IMAGE_HEIGHT, IMAGE_WIDTH, scale=(0.8, 1.0), p=0.5), # crop ngẫu nhiên


In [53]:
# Load CSV File
metadata = pd.read_csv(METADATA_CSV_PATH)

# Quick look
metadata.head()

,image,mask
0,10452_sat_08.jpg,10452_mask_08.jpg
1,10452_sat_18.jpg,10452_mask_18.jpg
2,111335_sat_00.jpg,111335_mask_00.jpg
3,111335_sat_01.jpg,111335_mask_01.jpg
4,111335_sat_02.jpg,111335_mask_02.jpg


In [54]:
# Add root path to image file names
metadata['image'] = [os.path.join(ROOT_IMAGE_DIR,filename) for filename in metadata['image']]

# Add mask path to image file names
metadata['mask']  = [os.path.join(ROOT_MASK_DIR,filename) for filename in metadata['mask']]

In [55]:
# Quick Check
metadata.head()

,image,mask
0,C:\Users\Admin\Deep Learning\Project\Forest Se...,C:\Users\Admin\Deep Learning\Project\Forest Se...
1,C:\Users\Admin\Deep Learning\Project\Forest Se...,C:\Users\Admin\Deep Learning\Project\Forest Se...
2,C:\Users\Admin\Deep Learning\Project\Forest Se...,C:\Users\Admin\Deep Learning\Project\Forest Se...
3,C:\Users\Admin\Deep Learning\Project\Forest Se...,C:\Users\Admin\Deep Learning\Project\Forest Se...
4,C:\Users\Admin\Deep Learning\Project\Forest Se...,C:\Users\Admin\Deep Learning\Project\Forest Se...


In [56]:
def load_dataset(
    image_paths: list, mask_paths: list, split_ratio: float = 0.2, 
    batch_size: int = BATCH_SIZE, shuffle: bool = True, 
    buffer_size: int = 1000, n_repeat: int = 1
) -> Union[Tuple[DataLoader, DataLoader], DataLoader]:
    '''
    This function loads the image and mask data from the provided file paths and creates a PyTorch DataLoader. 
    The function first creates space to store the image and mask data in numpy arrays. It then iterates over 
    each image and mask pair, loading them using the load_image_and_mask function and storing them in the numpy arrays.
    
    The function then creates a PyTorch TensorDataset using the numpy arrays. If shuffle is True, it shuffles 
    the dataset (via DataLoader). If split_ratio is not None, it splits the dataset into two parts with sizes 
    determined by the split_ratio, and converts them into batches of size batch_size with drop_last=True. 
    The two resulting DataLoaders are returned as a tuple.

    If split_ratio is None, the entire dataset is converted into batches of size batch_size with drop_last=True, 
    and the resulting DataLoader is returned.
    
    Args:
        image_paths: A list of strings, containing the file paths of the input images.
        
        mask_paths: A list of strings, containing the file paths of the corresponding mask images.
        
        split_ratio: A float value between 0 and 1, representing the ratio of data to be used for validation. 
                    If split_ratio is set to None, then no data will be split for validation.
                    
        batch_size: An integer, representing the batch size for the input data.
        
        shuffle: A boolean value indicating whether the data should be shuffled or not.
        
        buffer_size: (Kept for compatibility; in PyTorch, DataLoader shuffle does not use buffer_size).
        
        n_repeat: An integer, representing the total number of repetitions of the dataset (simulating tf.repeat).
    
    Returns:
        If split_ratio is not None, then the function returns a tuple of two PyTorch DataLoaders. 
        The first DataLoader contains the training data and the second DataLoader contains the validation data.
        
        If split_ratio is None, then the function returns a single PyTorch DataLoader containing the 
        input data batched for training.
    '''
    
    # Create space for storing the data.
    images = np.empty(shape=(len(image_paths), *IMAGE_SIZE), dtype=np.float32)
    masks  = np.empty(shape=(len(mask_paths), *MASK_SIZE),  dtype=np.float32)
    
    # Iterate over the data.
    index = 0
    for image_path, mask_path in tqdm(zip(image_paths, mask_paths), desc='Loading'):
        
        # Load the image and the mask.
        image, mask = load_image_and_mask(image_path=image_path, mask_path=mask_path)
        
        # Store the image and the mask.
        images[index] = image
        masks[index]  = mask
        
        # Increment the index.
        index += 1
    
    # Convert numpy arrays to torch tensors
    images = torch.tensor(images, dtype=torch.float32)
    masks  = torch.tensor(masks, dtype=torch.float32)

    # Create base dataset
    base_dataset = TensorDataset(images, masks)

    # Repeat dataset if needed
    if n_repeat > 1:
        base_dataset = ConcatDataset([base_dataset] * n_repeat)

    # Wrapper to apply augmentations
    class AugmentedDataset(torch.utils.data.Dataset):
        def __init__(self, subset, transform):
            self.subset = subset
            self.transform = transform
        def __len__(self):
            return len(self.subset)
        def __getitem__(self, idx):
            image, mask = self.subset[idx]
            image_np = (image.permute(1,2,0).numpy() * 255).astype(np.uint8)
            mask_np  = (mask.squeeze(0).numpy() * 255).astype(np.uint8)
            augmented = self.transform(image=image_np, mask=mask_np)
            img = augmented["image"]                          # Tensor CHW
            msk = augmented["mask"].unsqueeze(0).float() / 255.0
            return img, msk

    # Train/validation split
    if split_ratio is not None:
        keep_ratio = 1 - split_ratio
        data_1_len = int(keep_ratio * len(base_dataset))
        data_2_len = len(base_dataset) - data_1_len

        train_subset, val_subset = random_split(base_dataset, [data_1_len, data_2_len])

        train_ds = AugmentedDataset(train_subset, train_transform)
        val_ds   = AugmentedDataset(val_subset, val_transform)

        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=shuffle, drop_last=True)
        val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False, drop_last=True)

        print(f"Train samples: {len(train_ds)}, Validation samples: {len(val_ds)}")

        return train_loader, val_loader
    
    else:
        full_ds = AugmentedDataset(base_dataset, train_transform)
        full_loader = DataLoader(full_ds, batch_size=batch_size, shuffle=shuffle, drop_last=True)
        print(f"Full dataset samples: {len(full_ds)}")
        return full_loader

In [57]:
# Training and Testing Data
train_loader, test_loader = load_dataset(
    image_paths = metadata['image'],
    mask_paths = metadata['mask'],
    split_ratio = 0.1,
    shuffle = True,
    n_repeat = 3,
)

Loading: 5108it [00:14, 358.77it/s]


Train samples: 13791, Validation samples: 1533


In [58]:
print("*" * 100)
print(f"{' ' * 30}Training Data Size     : {len(train_loader.dataset)} samples "
      f"({len(train_loader)} batches of size {BATCH_SIZE})")
print(f"{' ' * 30}Test Data Size   : {len(test_loader.dataset)} samples "
      f"({len(test_loader)} batches of size {BATCH_SIZE})")
print("*" * 100)

****************************************************************************************************
                              Training Data Size     : 13791 samples (430 batches of size 32)
                              Test Data Size   : 1533 samples (47 batches of size 32)
****************************************************************************************************


In [ ]:
# Full training dataset size
full_train_size = len(train_loader.dataset)

# Split ratio
train_val_split = 0.1
valid_size = int(full_train_size * train_val_split)
train_size = full_train_size - valid_size

# Split dataset into train and validation
train_dataset, valid_dataset = random_split(train_loader.dataset, [train_size, valid_size])

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader   = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=True)

In [60]:
# Print dataset sizes
print("*" * 100)
print(f"{' ' * 30}Training Data Size   : {len(train_loader.dataset)} samples "
      f"({len(train_loader)} batches of size {BATCH_SIZE})")
print(f"{' ' * 30}Validation Data Size : {len(val_loader.dataset)} samples "
      f"({len(val_loader)} batches of size {BATCH_SIZE})")
print(f"{' ' * 30}Testing Data Size    : {len(test_loader.dataset)} samples "
      f"({len(test_loader)} batches of size {BATCH_SIZE})")
print("*" * 100)

****************************************************************************************************
                              Training Data Size   : 12412 samples (387 batches of size 32)
                              Validation Data Size : 1379 samples (43 batches of size 32)
                              Testing Data Size    : 1533 samples (47 batches of size 32)
****************************************************************************************************
